In [6]:
import re
import csv
from pathlib import Path
from collections import Counter

BASE = Path.cwd().parent
RAW = BASE / "Datasets" / "raw"
OUT = BASE / "Datasets" / "processed"
OUT.mkdir(parents=True, exist_ok=True)


LABELS = {
    "Biotechnology":    "Biotechnology & Applied Microbiology",
    "Chemistry":        "Chemistry",
    "CS_AI":            "Computer Science, Artificial Intelligence",
    "CS_InfoSystems":   "Computer Science, Information Systems",
    "Engineering_EE":   "Engineering, Electrical & Electronic",
    "Environmental":    "Environmental Sciences",
    "MaterialsScience": "Materials Science",
    "Mathematics":      "Mathematics",
    "Physics_Applied":  "Physics, Applied",
    "Psychology":       "Psychology",
}


DOI_PATTERN = re.compile(r"(10\.\d{4,9}/[^\s\],;]+)")

print(f"{len(LABELS)} target fields defined")
print("output will go to:", OUT)

10 target fields defined
output will go to: /Users/kalai/Documents/Dissertation/Datasets/processed


In [7]:
def parse_wos_file(path):
    
    records, current, current_tag = [], {}, None
    with open(path, encoding="utf-8-sig") as f:
        for line in f:
            line = line.rstrip("\n")
            if line.startswith("ER"):
                if current:
                    records.append(current)
                current, current_tag = {}, None
                continue
            if line.startswith("EF"):
                break
            match = re.match(r"^([A-Z][A-Z0-9]) (.*)$", line)
            if match:
                current_tag = match.group(1)
                current.setdefault(current_tag, []).append(match.group(2))
            elif line.startswith("   ") and current_tag:
                current[current_tag].append(line[3:])
    return records


def field(record, tag):
    
    return " ".join(record.get(tag, [])).strip()


def split_field(record, tag):
    
    return [x.strip() for x in field(record, tag).split(";") if x.strip()]


def matches_stem(categories, stem):
  
    return any(c == stem or c.startswith(stem + ",") for c in categories)


def reference_dois(record):
    
    found = set()
    for ref in record.get("CR", []):
        m = DOI_PATTERN.search(ref)
        if m:
            found.add(m.group(1).lower().rstrip("."))
    return sorted(found)

In [8]:


papers, duplicates = {}, 0

for path in sorted(RAW.glob("*.txt")):
    for record in parse_wos_file(path):
        ut = field(record, "UT")
        if not ut:
            continue                 
        if ut in papers:
            duplicates += 1
        else:
            papers[ut] = record

print(f"rows read:              10000")
print(f"unique papers:          {len(papers)}")
print(f"duplicate rows removed: {duplicates}")

rows read:              10000
unique papers:          8227
duplicate rows removed: 1773


In [9]:
rows, label_counts, unlabelled = [], Counter(), 0

for ut, record in papers.items():
    categories = split_field(record, "WC")
    labels = [name for name, stem in LABELS.items()
              if matches_stem(categories, stem)]

    if not labels:
        unlabelled += 1              
        continue

    for l in labels:
        label_counts[l] += 1

    refs = reference_dois(record)
    rows.append({
        "ut": ut,
        "doi": field(record, "DI").lower(),
        "year": field(record, "PY"),
        "title": field(record, "TI"),
        "abstract": field(record, "AB"),
        "journal": field(record, "SO"),
        "wos_categories": "; ".join(categories),
        "n_labels": len(labels),
        "labels": "|".join(labels),
        "n_refs": len(record.get("CR", [])),
        "ref_dois": "|".join(refs),
        **{f"lbl_{name}": int(name in labels) for name in LABELS},
    })

print(f"papers with at least one target label: {len(rows)}")
print(f"papers dropped (no target label):      {unlabelled}")

print("\npapers per field:")
for name in LABELS:
    print(f"  {label_counts[name]:>5}  {name}")

print("\nlabels per paper:")
for n, c in sorted(Counter(r["n_labels"] for r in rows).items()):
    print(f"  {n} label(s): {c}")

papers with at least one target label: 8192
papers dropped (no target label):      35

papers per field:
   1000  Biotechnology
   1656  Chemistry
   1035  CS_AI
   1011  CS_InfoSystems
   1435  Engineering_EE
    988  Environmental
   1384  MaterialsScience
   1000  Mathematics
   1004  Physics_Applied
   1015  Psychology

labels per paper:
  1 label(s): 5780
  2 label(s): 1488
  3 label(s): 924


In [10]:
out_path = OUT / "dataset.csv"

with open(out_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
    writer.writeheader()
    writer.writerows(rows)

print(f"written: {out_path}")
print(f"{len(rows)} rows, {len(rows[0])} columns")

written: /Users/kalai/Documents/Dissertation/Datasets/processed/dataset.csv
8192 rows, 21 columns
